In [ ]:
!pip install datasets gradio transformers jiwer tqdm
!pip install git+https://github.com/openai/whisper.git
# !pip install python-Levenshtein

In [ ]:
import requests
import jiwer
from datasets import load_dataset
import whisper
import random
from transformers import pipeline
import gradio as gr

import re
import string
from jiwer import wer
import os

import pandas as pd
from tqdm.auto import tqdm

In [ ]:
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^\w\s-]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [ ]:
dataset_vreme = load_dataset("iulik-pisik/audio_vreme", split="all", trust_remote_code=True)
dataset_vreme2 = load_dataset("iulik-pisik/audio_vreme", split="test", trust_remote_code=True)
dataset_vreme3 = load_dataset("iulik-pisik/audio_vreme", split="test+validation", trust_remote_code=True)

ip = dataset_vreme.filter(lambda example: example['name'] == "Iulia Parlea")
ls = dataset_vreme2.filter(lambda example: example['name'] == "Loredana Stefu")
fb = dataset_vreme2.filter(lambda example: example['name'] == "Florin Busuioc")
cs = dataset_vreme3.filter(lambda example: example['name'] == "Cosmin Stan")

ns = load_dataset("iulik-pisik/horoscop_neti", split="test", trust_remote_code=True)
ur = load_dataset("iulik-pisik/horoscop_urania", split="all", trust_remote_code=True)

Generating train split: 0 examples [00:00, ? examples/s]


Se citesc datele...: 0it [00:00, ?it/s]
Se citesc datele...: 1000it [00:00, 8479.97it/s]
Se citesc datele...: 2242it [00:00, 9976.81it/s] 


Generating test split: 0 examples [00:00, ? examples/s]


Se citesc datele...: 578it [00:00, 9471.06it/s]


Generating validation split: 0 examples [00:00, ? examples/s]

...


Se citesc datele...: 184it [00:00, 12538.41it/s]


Generating validation split: 0 examples [00:00, ? examples/s]


Se citesc datele...: 364it [00:00, 11643.56it/s]


Generating test split: 0 examples [00:00, ? examples/s]


Se citesc datele...: 184it [00:00, 14179.32it/s]


In [ ]:
# @title Definirea metadatelor pentru fiecare dataset
all_datasets = [
    {
        "dataset": ns,
        "category": "horoscope",
        "name": "Neti Sandu",
        "gender": "fem"
    },
     {
        "dataset": ur,
        "category": "horoscope",
        "name": "Urania",
        "gender": "fem"
    },
     {
        "dataset": fb,
        "category": "weather",
        "name": "Florin Busuioc",
        "gender": "masc"
    },
    {
        "dataset": cs,
        "category": "weather",
        "name": "Cosmin Stan",
        "gender": "masc"
    },
    {
        "dataset": ip,
        "category": "weather",
        "name": "Iulia Parlea",
        "gender": "fem"
    },
    {
        "dataset": ls,
        "category": "weather",
        "name": "Loredana Stefu",
        "gender": "fem"
    }
]

In [ ]:
def extract_filename(path):
    return path.split('/')[-1]

In [ ]:
model_name = "iulik-pisik/all_data_model_small"
folder_name = model_name.split("/")[1]
save_folder = f"/content/drive/MyDrive/licenta/outputs/{folder_name}"

if not os.path.exists(save_folder):
    os.mkdir(save_folder)

pipe = pipeline(model="iulik-pisik/all_data_model_small", task="automatic-speech-recognition", device=0)

def query(audio):
    text = pipe(audio)["text"]
    return text

config.json:   0%|          | 0.00/1.32k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.48M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


preprocessor_config.json:   0%|          | 0.00/339 [00:00<?, ?B/s]

In [ ]:
# import warnings

# warnings.filterwarnings("ignore", category=UserWarning)


In [ ]:
for dataset_info in all_datasets:
    dataset = dataset_info["dataset"]
    category = dataset_info["category"]
    name = dataset_info["name"]
    gender = dataset_info["gender"]

    # Inițializăm un DataFrame gol pentru fiecare dataset
    df_results = pd.DataFrame(columns=['filename', 'ground_truth', 'prediction', 'model', 'category', 'name', 'gender', 'wer', 'wer_nocase'])

    # Utilizăm tqdm pentru a afișa bara de progres
    for example in tqdm(dataset, desc=f"Processing {name}", unit="audio"):
        audio_path = example["path"]
        filename = audio_path.split('/')[-1]
        ground_truth = example["sentence"]
        prediction = query(audio_path)

        wer_value = wer(ground_truth, prediction) * 100
        wer_nocase = wer(preprocess_text(ground_truth), preprocess_text(prediction)) * 100

        # Utilizăm pandas.concat în loc de append
        new_row = pd.DataFrame([{
            'filename': filename, 'ground_truth': ground_truth, 'prediction': prediction,
            'model': model_name, 'category': category, 'name': name, 'gender': gender,
            'wer': wer_value, 'wer_nocase': wer_nocase
        }])
        df_results = pd.concat([df_results, new_row], ignore_index=True)

    name = name.replace(' ', '_').lower()
    csv_filename = os.path.join(save_folder, f"{name}.csv")
    df_results.to_csv(csv_filename, index=False)
    print(f"Results for {name} saved to {csv_filename}")

Processing Neti Sandu:   0%|          | 0/184 [00:00<?, ?audio/s]

Results for neti_sandu saved to /content/drive/MyDrive/licenta/outputs/all_data_model_small/neti_sandu.csv


Processing Urania:   0%|          | 0/184 [00:00<?, ?audio/s]

Results for urania saved to /content/drive/MyDrive/licenta/outputs/all_data_model_small/urania.csv


Processing Florin Busuioc:   0%|          | 0/231 [00:00<?, ?audio/s]

...

Results for cosmin_stan saved to /content/drive/MyDrive/licenta/outputs/all_data_model_small/cosmin_stan.csv


Processing Iulia Parlea:   0%|          | 0/261 [00:00<?, ?audio/s]

Results for iulia_parlea saved to /content/drive/MyDrive/licenta/outputs/all_data_model_small/iulia_parlea.csv


Processing Loredana Stefu:   0%|          | 0/256 [00:00<?, ?audio/s]

Results for loredana_stefu saved to /content/drive/MyDrive/licenta/outputs/all_data_model_small/loredana_stefu.csv


In [ ]:

# # Setează dimensiunea batch-ului
# batch_size = 8  # Ajustează în funcție de dimensiunea modelului și memoria GPU

# for dataset_info in all_datasets:
#     dataset = dataset_info["dataset"]
#     category = dataset_info["category"]
#     name = dataset_info["name"]
#     gender = dataset_info["gender"]

#     # Inițializăm un DataFrame gol pentru fiecare dataset
#     df_results = pd.DataFrame(columns=['filename', 'ground_truth', 'prediction', 'model', 'category', 'name', 'gender', 'wer', 'wer_nocase'])

#     # Preprocesează și colectează datele în batch-uri
#     num_batches = (len(dataset) + batch_size - 1) // batch_size  # Calculează numărul total de batch-uri
#     for i in tqdm(range(num_batches), desc=f"Processing {name}", unit="batch"):
#         batch = dataset.select(range(i * batch_size, min((i + 1) * batch_size, len(dataset))))
#         audio_paths = [ex["path"] for ex in batch]
#         ground_truths = [ex["sentence"] for ex in batch]
#         predictions = pipe(audio_paths)

#         for audio_path, ground_truth, prediction in zip(audio_paths, ground_truths, predictions):
#             filename = os.path.basename(audio_path)
#             wer_value = wer(ground_truth, prediction['text']) * 100
#             wer_nocase = wer(preprocess_text(ground_truth), preprocess_text(prediction['text'])) * 100

#             # Adaugă rezultatele în DataFrame
#             new_row = {'filename': filename, 'ground_truth': ground_truth, 'prediction': prediction['text'],
#                        'model': model_name, 'category': category, 'name': name, 'gender': gender,
#                        'wer': wer_value, 'wer_nocase': wer_nocase}
#             df_results = pd.concat([df_results, pd.DataFrame([new_row])], ignore_index=True)

#     # Salvăm rezultatele într-un CSV
#     name = name.replace(' ', '_').lower()
#     csv_filename = os.path.join(save_folder, f"{name}.csv")
#     df_results.to_csv(csv_filename, index=False)
#     print(f"Results for {name} saved to {csv_filename}")
